©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、PyTorchを用いて畳み込みニューラルネットワーク（CNN）を実装します。CIFAR-10データセットから鳥と飛行機の2クラス分類を行い、畳み込み・パディング・プーリングの使い方、nn.Moduleによるモデル構築、学習ループの実装までCNNの一連の流れを習得します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
%%capture
!pip uninstall matplotlib -y
!pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

!pip uninstall torch -y
!pip install torch==2.7.0

!pip uninstall torchvision -y
!pip install torchvision==0.22.0

# CNN　コード演習

「画像の深層学習」と言えばCNNというくらいメジャーな手法です。CNNはConvolutional Neural Networkの頭文字を取ったもので、ニューラルネットワークに「畳み込み」という操作を導入したも
のになります。

本演習では、CNNに必要な技術を確認した上で、最終的にCNNモデルを実装します。

---

目次
1.   事前準備
  *  ライブラリのインポート
  *  画像データセットを構築
2.   畳み込みの使用方法
  *   境界のパディング
  *   畳み込みを用いた特徴量の検出
  *   深さとプーリングの詳細
  *   nn.Sequential型の限界
3.   nn.Moduleを継承したモデルの構築方法
  *   nn.Moduleのネットワーク
  *   PyTorchがパラメータとサブモジュールを追跡する原理
  *   functional API
4.   畳み込みニューラルネットワークの学習
  *   正解率の測定
  *   モデルの保存と読み込み
  *   GPU上での訓練
5.   複雑なモデルの作り方
  *   記憶容量の追加 - 幅
  *   モデルの収束と汎化の補助：正則化
  *   単一の入力に依存しすぎない：ドロップアウト
  *   バッチ正規化
  *   レイヤー正規化
  *   Instance正規化
  *   複雑な構造体を学習するためにより深く：深さ
  *   スキップ接続
  *   ResBlockを作り、非常にディープなモデルを構築する








# 1. 事前準備

### ライブラリのインポート

In [ ]:
# 必要なライブラリのインポート
from matplotlib import pyplot as plt
import numpy as np
import collections
import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

### 画像データセットの構築

CIFAR-10のデータセットから全ての鳥と飛行機を選び出し、鳥と飛行機を見分けるニューラルネットワークを構築します。

In [ ]:
# 分類クラスの名前
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']  # 全10クラス

In [ ]:
# torchvisionライブラリを使って、CIFAR-10の訓練用およびテスト用データをダウンロード
# ダウンロード時に、画像をTensorに変換し、平均と標準偏差で正規化を行う
from torchvision import datasets, transforms

data_path = '../data-unversioned/'

# 訓練データの読み込みと前処理
cifar10 = datasets.CIFAR10(
    data_path, train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),  # 各RGBチャネルの平均
                             (0.2470, 0.2435, 0.2616))  # 各RGBチャネルの標準偏差
    ]))

# テストデータ（検証用）の読み込みと前処理
cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

# CIFAR-10から「飛行機（label=0）」と「鳥（label=2）」のデータのみを抽出
# また、ラベルを連続する0, 1にマッピングし直す（0: airplane, 2: bird → 0: airplane, 1: bird）
label_map = {0: 0, 2: 1}
class_names = ['airplane', 'bird']  # 以降は2クラス分類

# 訓練データ：対象クラスのみフィルタし、マッピングを適用
cifar2 = [(img, label_map[label]) for img, label in cifar10 if label in [0, 2]]  # list化で簡易Dataset互換

# 検証データ：同様にフィルタとマッピングを適用
cifar2_val = [(img, label_map[label]) for img, label in cifar10_val if label in [0, 2]]

# cifar2オブジェクトは、要素の取得（__getitem__）と長さの取得（__len__）が可能なため、
# PyTorchのDataset互換インターフェースとしてそのまま使用可能
# これは、簡易的にサブクラスを作らずに二値分類用データセットを構築する方法
# ただし、より複雑な前処理や構造が必要な場合は、正式なDatasetサブクラスを作る必要がある点に注意

# 2. 畳み込みの使用方法

鳥と飛行機の分類のような画像認識では、PyTorchの torch.nn.Conv2d を使って画像に畳み込み処理を行います。Conv2d は、入力チャネル数（例：RGBなら3）、出力チャネル数（検出したい特徴の数）、カーネルサイズ（例：3×3）を指定して使います。

たとえば、最初の畳み込み層では、RGB画像（3チャネル）を16チャネルに変換し、画像からさまざまな特徴を抽出します。出力チャネル数を増やすとネットワークの表現力が高まりますが、必ずしもすべての出力が有用になるとは限りません。

カーネルサイズは通常、すべての方向で同じにし、kernel_size=3 で 3×3 のカーネルを意味します。Pythonでは (3, 3) のタプルでも指定可能です。今回は、サイズをそろえた標準的な畳み込みに焦点を当てて学びます。

In [ ]:
# 簡易的にkernel_size=3と指定する代わりに、
# kernel_size=(3, 3)という、
# 出力時に表示されるのと同じタプル形式で渡すこともできます
conv = nn.Conv2d(3, 16, kernel_size=3)
conv

In [ ]:
# weightとbiasのshapeの確認
conv.weight.shape, conv.bias.shape

In [ ]:
# nn.Conv2dは入力として、BxCxHxWを想定している
# 1枚の入力画像に対してconvモジュールを呼び出す場合、0番目にバッチ次元をunsqueezeで追加する必要があります。
img, _ = cifar2[0]
output = conv(img.unsqueeze(0))
# ０番目に１が追加されていることを確認する
img.unsqueeze(0).shape, output.shape

In [ ]:
# 畳み込みの結果、画像が小さくなる
plt.figure(figsize=(10, 4.8))
ax1 = plt.subplot(1, 2, 1)
plt.title('output')
plt.imshow(output[0, 0].detach(), cmap='gray')  # 可視化のため1chだけ参照
plt.subplot(1, 2, 2, sharex=ax1, sharey=ax1)
plt.imshow(img.mean(0), cmap='gray')  # 入力の平均グレースケール
plt.title('input')
plt.show()

## 境界のパディング

出力画像が入力画像よりも小さくなる現象は、画像の境界に対して行う畳み込み処理の副次的な結果です。畳み込みカーネルを3x3近傍のピクセルの重み付けされた合計として適用すると、全ての方向に隣接するピクセルが存在することが必要になります。i00の位置にいるとしたら、右方向と下方向にしかピクセルが存在しません。デフォルトでは、PyTorchは畳み込みカーネルを入力画像内でスライドさせ、width - kernel_width + 1の横位置と縦位置までを取得します。奇数のカーネルでは、この結果、畳み込みカーネルの幅の半分（今回の場合、3//2 = 1）だけ、各境界の辺が小さくなった画像が得られます。これが各次元で2ピクセル分が失われた理由です。

しかしPytorchでは畳み込み時に、境界領域の周りに値がゼロになる架空のピクセルを作成して、画像をパディングすることができます。

今回のケースでは、kernel_size=3のときにpadding=1を指定すると、i00の上と左に隣接するピクセルが余分に存在することになります。その結果、出力は入力と全く同じサイズになりました。

In [ ]:
# パディングによって画像サイズを維持する
conv = nn.Conv2d(3, 1, kernel_size=3, padding=1)
output = conv(img.unsqueeze(0))
img.unsqueeze(0).shape, output.shape

In [ ]:
# なお、重みとバイアスの大きさはパディングの有無によって変化しない
conv.weight.shape, conv.bias.shape

## 畳み込みを用いた特徴量の抽出

畳み込みにおいて重みとバイアスは、nn.Linearの場合と同様に、バックプロパッゲーションによって学習されるパラメータと説明しました。ですが、ここでは、手動で重みを設定した畳み込みを用いて何が起こるか確認しておきましょう。

まず問題を簡単にするためにバイアスはゼロとし、出力の各ピクセルが隣接するピクセルの平均になるように重みに一定の値を設定してみます。3x3の近傍それぞれについて、以下のような処理をします。

In [ ]:
# 重みとバイアスを手動で設定し、畳み込みの効果を確認する
with torch.no_grad():
    conv.bias.zero_()

with torch.no_grad():
    conv.weight.fill_(1.0 / 9.0)  # 平均化フィルタ（全要素1/9）

# 畳込みの結果、ピクセル同士の特徴がまとめられ、ぼやけた画像が表示される
output = conv(img.unsqueeze(0))
plt.imshow(output[0, 0].detach(), cmap='gray')
plt.show()

In [ ]:
# 手作りの畳み込みカーネルを適応した後に描画した鳥の画像
# 全体的に垂直方向のエッジが検出されていることが分かる
conv = nn.Conv2d(3, 1, kernel_size=3, padding=1)
with torch.no_grad():
    conv.weight[:] = torch.tensor([[-1.0, 0.0, 1.0],
                                  [-1.0, 0.0, 1.0],
                                  [-1.0, 0.0, 1.0]])
    conv.bias.zero_()

output = conv(img.unsqueeze(0))
plt.imshow(output[0, 0].detach(), cmap='gray')
plt.show()

## 深さとプーリングの詳細

ここまでの構成でも分類は可能ですが、まだ重要な概念が残っています。それは「大きな画像や複雑な構造にどう対応するか」です。

畳み込み層により、全結合層では得られなかった局所性と移動不変性が実現されました。特に、3×3や5×5など小さなカーネルを使うことで、画像内の細かなパターン（局所的特徴）を捉えることができます。

しかし、大きな画像ではどうでしょうか？
本当にすべての重要な特徴が3〜5ピクセルの範囲内に収まっているのでしょうか？実際はそうではありません。特にCIFAR-10のような画像でも、飛行機の全体的な形や鳥の姿勢といった大域的な特徴は小さなカーネルだけでは捉えきれません。

このとき、1つの選択肢は大きなカーネルを使うことです。ただし、画像全体と同じサイズのカーネル（例：32×32）を使ってしまうと、全結合層と同じ状態になってしまい、畳み込みの利点を失います。

そこでより良い選択肢は、小さなカーネルを重ねて深い層を構成し、途中でプーリングを挟んでダウンサンプリングすることです。これにより、局所的な情報から段階的に大域的な特徴を捉えられるようになります。

In [ ]:
# マックスプーリングは、nn.MaxPool2dとして提供されています
# このモジュールは、プーリング処理を行う近傍のサイズを入力として受け取ります
# 画像を半分にダウンサンプリングしたい場合は、サイズに2を使用します
pool = nn.MaxPool2d(2)
output = pool(img.unsqueeze(0))
img.unsqueeze(0).shape, output.shape

## nn.Sequential型の限界

CNNでは、最初の畳み込み層でRGB画像（3チャネル）を16チャネルに変換し、Tanh活性化とMaxPoolingによって32×32→16×16に縮小します。さらに8チャネルに畳み込み、同様にTanhとプーリングで8×8の特徴マップを得ます。

ここまではnn.Sequentialで処理できますが、得られた出力はまだ2次元の画像（8チャネル×8×8）であり、そのままでは分類に必要な2クラスの確率（飛行機 or 鳥）を出すことができません。

そのため、2次元の出力を1次元ベクトルにフラット化し、全結合層でクラスごとのスコアを出力させる必要があります。これがnn.Sequentialだけでは完結しにくい理由です。

In [ ]:
# 不完全モデル
model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 入力(3ch)→16ch, 3x3畳み込み（padding=1で空間サイズ維持: 32→32）
            nn.Tanh(),  # 活性化：tanh
            nn.MaxPool2d(2),  # 2x2プーリングで空間サイズ半減: 32→16
            nn.Conv2d(16, 8, kernel_size=3, padding=1),  # 16ch→8ch, 3x3（空間 16→16）
            nn.Tanh(),
            nn.MaxPool2d(2),  # 空間 16→8
            nn.Flatten(),  # (B, 8, 8, 8) に相当するテンソルを1次元へ
            # ここで１次ベクトルへの変換をする必要がある
            nn.Linear(8 * 8 * 8, 32),  # 入力32→16→プーリング2回で 8×8×8
            nn.Tanh(),
            nn.Linear(32, 2))  # 出力は2クラスのロジット

In [ ]:
# パラメータの数だけ確認する
numel_list = [p.numel() for p in model.parameters()]  # 各パラメータテンソルの要素数（重み+バイアス）を列挙
sum(numel_list), numel_list  # 合計パラメータ数と層ごとの内訳を確認

# 3. nn.Moduleを継承したモデルの構築方法

ニューラルネットを構築する際に、既製のモジュールにはない処理を行いたい状況に出くわすこともあるでしょう。単純な例としてはテンソルの変形処理が挙げられます。

次から次へとレイヤを適用するだけでなく、より複雑なことを行うモデルを構築したい場合は、nn.Sequentialを使うのではなく、柔軟性の高いモジュールを用意する必要があります。PyTorchではnn.Moduleをサブクラス化することで、モデル内で任意の処理を行うことができます。

## nn.Moduleのネットワーク

サブモジュールとして独自のネットワークを構築させましょう。まず、先ほどnn.Sequentialで使用したnn.Conv2dやnn.Linearなどの、全てのサブモジュールをコンストラクタ内でインスタンス化します。そして、それらのインスタンスをforward関数内で次から次へと使用します。

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # 3→16, 3x3 SAME風
        self.act1 = nn.Tanh()
        self.pool1 = nn.MaxPool2d(2)  #~ 32→16
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)  # 16→8
        self.act2 = nn.Tanh()
        self.pool2 = nn.MaxPool2d(2) #~ 16→8
        self.fc1 = nn.Linear(8 * 8 * 8, 32)  # Flatten後 (C=8,H=8,W=8) を32へ
        self.act3 = nn.Tanh()
        self.fc2 = nn.Linear(32, 2)  # 2クラスのロジット出力

    def forward(self, x):
        out = self.pool1(self.act1(self.conv1(x)))  # (B,3,32,32)→Conv→tanh→Pool→(B,16,16,16)
        out = self.pool2(self.act2(self.conv2(out)))  # →Conv→tanh→Pool→(B,8,8,8)
        # nn.Sequentialではできなかったテンソルの変形処理
        out = out.view(-1, 8*8*8)  # Flatten
        out = self.act3(self.fc1(out))
        out = self.fc2(out)
        return out

## PyTorchがパラメータとサブモジュールを追跡する原理

nn.Moduleのインスタンスをnn.Moduleの属性変数に代入すると、モジュールが自動的にサブモジュールとして登録されます。

nn.Moduleのサブクラスでは、任意のメソッドを呼び出すことが可能なので、ユーザが何もしなくても、Netがサブモジュールのパラメータにアクセスできるようになります。

In [ ]:
model = Net()
numel_list = [p.numel() for p in model.parameters()]  #~ パラメータ数を再チェック
sum(numel_list), numel_list  #~ 合計と内訳

## functional API

Netが訓練中にパラメータを管理できるように、nn.Linearとnn.Conv2dにおいてnnモジュールを使い続けることは合理的です。しかし、プーリングや活性化に関しては、パラメータを保有していないため、問題なくfunctional版に変更することが可能です。

In [ ]:
# 必要なモジュールをインポート
import torch.nn.functional as F

# 畳み込みニューラルネットワークの定義
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # 1つ目の畳み込み層: 3チャンネルの入力を受け取り、16チャンネルの出力を生成
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)

        # 2つ目の畳み込み層: 16チャンネルの入力を受け取り、8チャンネルの出力を生成
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)

        # 1つ目の全結合層: 8*8*8サイズの入力を受け取り、32サイズの出力を生成
        self.fc1 = nn.Linear(8*8*8, 32)

        # 2つ目の全結合層: 32サイズの入力を受け取り、2サイズの出力を生成（最終的なクラス分類のため）
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # 1つ目の畳み込み層を適用し、tanh活性化関数を使用した後、2x2のmax poolingを適用
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)

        # 2つ目の畳み込み層を適用し、tanh活性化関数を使用した後、2x2のmax poolingを適用
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)

        # 2Dテンソルを1Dテンソルに変換
        out = out.view(-1, 8 * 8* 8)

        # 1つ目の全結合層を適用し、tanh活性化関数を使用
        out = torch.tanh(self.fc1(out))

        # 2つ目の全結合層を適用して出力を得る
        out = self.fc2(out)

        return out

上記の実装は、前回実装したNetと完全に同じものを示しながら、はるかに簡潔になっています。しかし、依然としてコンストラクタにおいて初期化する必要のあるパラメータを持つモジュールはインスタント化する必要がある点に注意してください。

In [ ]:
# # なお、数内のハードコーデイングを避けたい場合は、initにパラメータを渡すことができる。

# class Net(nn.Module):
#     def __init__(self, n_chans1=16):
#         super().__init__()
#         self.n_chans1 = n_chans1
#         self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3,
#                                padding=1)
#         self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
#         self.fc2 = nn.Linear(32, 2)

#     def forward(self, x):
#         out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
#         out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
#         out = out.view(-1, 8 * 8 * self.n_chans1 // 2)
#         out = torch.tanh(self.fc1(out))
#         out = self.fc2(out)
#         return out

In [ ]:
model = Net()
model(img.unsqueeze(0))  # 形状・実行テスト用

# 4. 畳み込みニューラルネットワークの学習

CPU上で動作するので、必要に応じて「GPU上での訓練」まで、実行をスキップしてください。

In [ ]:
# CPUで実行したい場合はコメントアウトを外す。

import datetime # Pythonに含まれているdatetimeモジュールを使用

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    # 0からでなく1からn_epochsまで数字を振ったエポックに対するループ
    for epoch in range(1, n_epochs + 1):
        # エポックごとに損失を初期化
        loss_train = 0.0
        # バッチ単位でのループ
        for imgs, labels in train_loader:
            # フォーワード処理
            # モデルに画像を投入して、出力を得る
            outputs = model(imgs)
            # 損失を計算する
            loss = loss_fn(outputs, labels)
            # 勾配を初期化
            optimizer.zero_grad()
            # バックワード処理
            # 勾配を計算する
            loss.backward()
            # モデルを更新
            optimizer.step()
            # 現在のエポックの損失に加算
            loss_train += loss.item()
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(),
                epoch,
                (loss_train / len(train_loader))
            ))

In [ ]:
# CPUで実行したい場合はコメントアウトを外す。

# DataLoaderがcifar2のデータセットのサンプルをバッチ化
# また、データセットからサンプルを取得する順番をランダムにシャッフルする
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

# ネットワークのインスタンス化
model = Net()
# これまでと同じ確率的勾配降下のオプティマイザを使用し…
optimizer = optim.SGD(model.parameters(), lr=1e-2)
# クロスエントロピー誤差を使用します
loss_fn = nn.CrossEntropyLoss()
# 先ほど定義した訓練ループを呼び出します
training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader=train_loader)   # 学習データローダー

## 正解率の測定

損失よりも解釈しやすい評価指標を得るために、訓練データセットと検証データセットの正解率を確認します。

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=False)

def validate(model, train_loader, val_loader):
    for name, loader in [('train', train_loader), ('val', val_loader)]:
        correct = 0
        total = 0
        # 勾配の情報は必要ない
        with torch.no_grad():
            for imgs, labels in loader:
                outputs = model(imgs)
                # 出力で最大値をもつインデックスを取得
                _, predicted = torch.max(outputs, dim=1)
                # サンプル数を数え、totalをバッチサイズ分増やす
                total += labels.shape[0]
                # 位置する要素数を取得する
                correct += int((predicted == labels).sum())

            print('Accuracy {}:{:.2f}'.format(name, correct / total))

In [ ]:
# CPUで実行したい場合はコメントアウトを外す。

# validate(model, train_loader, val_loader)

## モデルの保存と読み込み

今のところモデルの性能には満足しているため、モデルを保存しておきたいところです。簡単にできます。次のように保存します。

In [ ]:
# モデル名
model_name = 'bird_vs_airplanes.pt'

In [ ]:
# 保存(モデルの重みを保存しており、構造は保持していないことに注意)
torch.save(model.state_dict(), data_path + model_name)

In [ ]:
# モデルを保存した時とモデルの状態を読み込む時で、
# Netの定義を変えてはいけない点に注意
loaded_model = Net()
loaded_model.load_state_dict(torch.load(data_path + model_name))  # CPUで読み込み

## GPU上での訓練

訓練の実行をGPUに移すことで訓練を速く行えます。toメソッドを利用することで、データローダから取得したテンソルをGPUに移すことができ、その後はGPU上で自動的に演算が行われます。ただし、モデルのパラメータもGPU上に移す必要があります。幸い、nn.Moduleには全てのパラメータをGPU上に移動する（引数dtypeを与えた場合はキャストも行える）.to関数が備わっています。

また、GPUが利用可能であれば、移動可能なものはGPUに移動するスタイルが良いと考えられています。良いコーディングの設計としては、torch.cuda.is_availableの返り値によってdevice変数に値を設定することです。

In [ ]:
device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('cpu'))  # CUDA(GPU)が使えるならGPUデバイスを、使えないならCPUデバイスを選択してdeviceに入れる
print(f"Training on device {device}.")  # 実際にどっちを使うことになったかを表示（デバッグ・確認用）

In [ ]:
# GPUでも利用できるように関数を変更する
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            # GPUが使える場合、GPUにデータを送る
            if torch.cuda.is_available():
                device = torch.device('cuda')
                imgs = imgs.to(device=device)       # 画像をGPUに移動
                labels = labels.to(device=device)   # ラベルをGPUに移動

            # それ以外の処理は同じ
            # モデルを使って予測を行う
            outputs = model(imgs)

            # 予測と実際のラベルとの間の損失を計算
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()     # オプティマイザの勾配をリセット
            loss.backward()           # バックプロパゲーションを用いて勾配を計算
            optimizer.step()          # オプティマイザのステップでパラメータを更新

            loss_train += loss.item()   # トータルの損失を更新

        # エポック1または10の倍数のエポックでトレーニングの損失を表示
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))

In [ ]:
# GPUで学習させる
device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('cpu'))

# ネットワークのインスタンスを作成し、指定されたデバイスに移動
model = Net().to(device=device)

# SGD（確率的勾配降下法）オプティマイザを使用し、学習率を0.01に設定
optimizer = optim.SGD(model.parameters(), lr=1e-2)

# クロスエントロピー損失関数を選択
loss_fn = nn.CrossEntropyLoss()

# トレーニングループの関数を呼び出し、トレーニングを開始
training_loop(
    n_epochs = 100,               # 100エポックでトレーニング
    optimizer = optimizer,        # 上で定義したSGDオプティマイザを使用
    model = model,                # 上で定義したモデルを使用
    loss_fn = loss_fn,            # クロスエントロピー損失関数を使用
    train_loader = train_loader,  # トレーニングデータのローダー
)

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=False)

def validate(model, train_loader, val_loader):
    # 学習データ(train)と検証データ(val)の両方で精度(accuracy)を測る関数
    # model はすでに学習済み or 現在学習中のネットワーク
    for name, loader in [('train', train_loader), ('val', val_loader)]:
        # まず 'train' データで評価し、そのあと 'val' データで評価する
        correct = 0
        total = 0
        # 勾配の情報は必要ない
        with torch.no_grad():
            for imgs, labels in loader:
              # GPUが使える場合、GPUにデータを送る
              if torch.cuda.is_available():
                  device = torch.device('cuda')
                  imgs = imgs.to(device=device)
                  labels = labels.to(device=device)

              outputs = model(imgs)
              # 出力で最大値をもつインデックスを取得
              _, predicted = torch.max(outputs, dim=1)
              # サンプル数を数え、totalをバッチサイズ分増やす
              total += labels.shape[0]
              # 位置する要素数を取得する
              correct += int((predicted == labels).sum())

            print('Accuracy {}:{:.2f}'.format(name, correct / total))

In [ ]:
# モデルのバリデーションを行う関数を呼び出す
# この関数は、トレーニングデータとバリデーションデータの両方でモデルのパフォーマンスを評価するために使用される
validate(model, train_loader, val_loader)

In [ ]:
# なお重みを読み込む際に、学習したデバイスに戻そうとするため、読み込む際には工夫が必要
# torch.load()メソッドで読み込み時にデバイス情報を上書きするようにする
loaded_model = Net().to(device=device)
loaded_model.load_state_dict(torch.load(data_path
                                        + model_name,
                                        map_location=device))  # ここで読み込んだstate_dict中の重みTensorを全部 device (CPU or GPU) にマップさせる

# 5. 複雑なモデルの作り方

デファクトスタンダードであり、最もシンプルなモデルであるnn.Moduleのサブクラスとしてモデルを構築しました。そして、そのモデルを訓練することに成功し、モデルを訓練する際にGPUを使う方法について解説してきました。以上により、畳み込みニューラルネットワークのフォワード処理を構築し、画像を分類できるように訓練して成功するところまで到達しました。次はどうするのか、という疑問が湧いてくるかもしれません。

本節の目的は、PyTorchで実装を始められるよう、概念的なテクニックを正確に解説することにあります。

## モデル容量の追加 - 幅

各層のパラメータを大きくすることで、モデルの容量を大きくする

In [ ]:
class NetWidth(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 入力(3ch)→32ch、3x3畳み込み（パディング1で空間サイズは維持）
        # 8→16に増加させる
        self.conv2 = nn.Conv2d(32, 16, kernel_size=3, padding=1)  # 32ch→16ch、3x3（同じくサイズ維持）
        self.fc1 = nn.Linear(16 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2)  # 2クラス分類のロジット出力

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)  # conv1→tanh→2x2プーリング
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        # 増加させた分、変更が必要
        out = out.view(-1, 16 * 8 * 8)
        out = torch.tanh(self.fc1(out))  # 全結合→tanh
        out = self.fc2(out)  # ロジット2次元（Softmaxは損失/評価側で実施）
        return out

In [ ]:
model = NetWidth().to(device=device)  # ネットワークをGPU/CPUへ転送
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

validate(model, train_loader, val_loader) # 学習後の精度を train/val で評価

## モデルの収束と汎化の補助：正則化

訓練を安定化させる1つ目の手法は、損失に正則化項を加えることです。この項は、モデルの重みが自身の値に応じて小さくさせるように設定することで、訓練によってそれらの重みが大きくなっていくことを抑制しています。大きな値を持つ重みへのペナルティとも言えます。

In [ ]:
# l2正則化を加える
def training_loop_l2reg(n_epochs, optimizer, model, loss_fn,
                        train_loader):
    # L2正則化項を手作業で足し込む学習ループ
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)  # データをデバイスへ
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)  # もとのクロスエントロピー損失
            # 正則化に必要な項目を加える
            # 正則化を加える度合いを与える
            l2_lambda = 0.001  #~ 正則化の強さ（ハイパーパラメータ）
            # L1正則化の場合は、累乗pow(2.0)を絶対値abs()に置き換える
            l2_norm = sum(p.pow(2.0).sum()
                          for p in model.parameters())  # すべてのパラメータのL2ノルム平方和
            # 損失に正則化項をたす
            loss = loss + l2_lambda * l2_norm  # 合成損失

            optimizer.zero_grad()  # 勾配リセット
            loss.backward()  # 逆伝播
            optimizer.step()  # パラメータ更新

            loss_train += loss.item()
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))  # エポック平均損失の表示（正則化込み）

In [ ]:
# L1正則化を加える場合
def training_loop_l1reg(n_epochs, optimizer, model, loss_fn,
                        train_loader):
    # L1正則化項（スパース化傾向）が必要な場合の学習ループ
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
            # 正則化に必要な項目を加える
            # 正則化を加える度合いを与える
            l1_lambda = 0.001  # 正則化係数
            # L1正則化の場合は、絶対値abs()を総和sum()に置き換える
            l1_norm = sum(p.abs().sum()
                          for p in model.parameters())  # |w| の総和
            # 損失に正則化項をたす
            loss = loss + l1_lambda * l1_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item()
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))

In [ ]:
# 正則化の効果を確認
model = Net().to(device=device)  # ベースモデル
optimizer = optim.SGD(model.parameters(), lr=1e-2) # 最適化はSGD
loss_fn = nn.CrossEntropyLoss()  # 損失関数にはクロスエントロピー損失を使用

training_loop_l2reg(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

validate(model, train_loader, val_loader)  # L2追加学習後の評価

In [ ]:
# 同様のことがSGDのweight_decay引数に数値を与えることで可能（デフォルトは０）
model = NetWidth().to(device=device)
optimizer = optim.SGD(model.parameters(), lr=1e-2, weight_decay=0.001)  # L2相当をoptimizer側で実現
loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

validate(model, train_loader, val_loader)

## ドロップアウト

nn.Dropoutモジュールを利用します。
- 非線形な活性化関数とその後ろの線形モジュール、もしくは畳み込みモジュールの間に加えます。
- 引数には、ドロップする確率を与える必要があります。
- 畳込みを利用する場合は、入力チャネル全体をドロップできるnn.Dropout2dやnn.Dropout3dなどを使うことが望ましいです。

なお、ドロップアウトは一般に訓練中にのみ有効なため、予測や検証の際には、ドロップアウトはスキップされることになります。

この挙動は、Dropoutモジュールのtrainプロパティで制御されており、model.train()とmodel.eval()で切り替えることができます。

In [ ]:
class NetDropout(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)  # Conv1: 3→n_chans1
        # ドロップアウトを定義（ドロップする確率を与える）
        self.conv1_dropout = nn.Dropout2d(p=0.4)  # 特徴マップ単位でランダムに無効化
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3,
                               padding=1)  # Conv2: n_chans1→n_chans1//2
        # ドロップアウトを定義（ドロップする確率を与える）
        self.conv2_dropout = nn.Dropout2d(p=0.4)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)  # 2回プーリング後の (C= n_chans1//2, H=W=8) を想定
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)  # Conv1→tanh→Pool
        # ドロップアウトを追加
        out = self.conv1_dropout(out)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        # ドロップアウトを追加
        out = self.conv2_dropout(out)
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)  # Flatten
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
# ドロップアウトの効果を確認

# ドロップアウトを持つネットワークのインスタンスを作成し、指定されたデバイスに移動
model = NetDropout(n_chans1=32).to(device=device)

# SGD（確率的勾配降下法）オプティマイザを使用し、学習率を0.01に設定
optimizer = optim.SGD(model.parameters(), lr=1e-2)

# クロスエントロピー損失関数を選択
loss_fn = nn.CrossEntropyLoss()

# モデルをトレーニングモードに設定 (ドロップアウトなどの層が動的に動作する)
model.train()

# トレーニングループの関数を呼び出し、トレーニングを開始
training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

model.eval()  # 評価モード（Dropout無効、BNは推論統計）
validate(model, train_loader, val_loader)

## バッチ正規化

入力次元に応じて、バッチ正規化として、nn.BatchNorm1d、nn.BatchNorm2d、nn.BatchNorm3dがある。バッチ線形変化層や活性化層のあとに入れるのが一般的とされる。

In [ ]:
class NetBatchNorm(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()

        # 初期化するチャネル数を設定
        self.n_chans1 = n_chans1

        # 1つ目の畳み込み層と、その後に続くバッチ正規化層
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)  # 3→n_chans1
        self.conv1_batchnorm = nn.BatchNorm2d(num_features=n_chans1)  # ミニバッチ平均/分散で正規化

        # 2つ目の畳み込み層と、その後に続くバッチ正規化層
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3,
                               padding=1)  # n_chans1→n_chans1//2
        self.conv2_batchnorm = nn.BatchNorm2d(num_features=n_chans1 // 2)

        # 全結合層
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)  # Flatten
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # 1つ目の畳み込み層を適用し、バッチ正規化を行った後、tanh活性化関数とmax poolingを適用
        out = self.conv1_batchnorm(self.conv1(x))  # BNは内部でaffine(γ,β)適用
        out = F.max_pool2d(torch.tanh(out), 2)

        # 2つ目の畳み込み層を適用し、バッチ正規化を行った後、tanh活性化関数とmax poolingを適用
        out = self.conv2_batchnorm(self.conv2(out))
        out = F.max_pool2d(torch.tanh(out), 2)

        # 2Dテンソルを1Dテンソルに変換して全結合層に適用
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)  # Flatten
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)

        return out

## レイヤー正規化

全チャンネルに跨って平均・分散をとる。
nn.LayerNorm()が利用可能

In [ ]:
# Layer Normalization を使った畳み込みニューラルネットワークの定義
class NetLayerNorm(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1  # 最初の畳み込み層の出力チャネル数

        # 入力はRGB画像（3チャネル）、出力は指定されたチャネル数（デフォルト32）
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        # Conv2dの出力に対して LayerNorm を適用
        self.conv1_layernorm = nn.LayerNorm(normalized_shape=n_chans1)

        # 2層目の畳み込み：チャネル数を半分に削減
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        # 2層目のLayerNorm（出力チャネル数に対応）
        self.conv2_layernorm = nn.LayerNorm(normalized_shape=n_chans1 // 2)

        # Flatten後、全結合層で32次元へ変換
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        # 最終出力は2クラス（例：飛行機 or 鳥）
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # 畳み込み → LayerNorm → tanh → MaxPooling（2×2）
        out = self.conv1_layernorm(self.conv1(x))
        out = F.max_pool2d(torch.tanh(out), 2)

        # 2層目も同様に：Conv → LayerNorm → tanh → MaxPooling
        out = self.conv2_layernorm(self.conv2(out))
        out = F.max_pool2d(torch.tanh(out), 2)

        # 特徴マップを1次元に変換（Flatten）
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)

        # 全結合層 + tanh活性化
        out = torch.tanh(self.fc1(out))

        # 出力層（ロジット出力、Softmaxは外部で使用）
        out = self.fc2(out)

        return out


## Instance正規化

各チャンネル独立に画像の縦横方向についてのみ平均・分散を取る。nn.InstanceNorm2d()が利用可能。

In [ ]:
# Instance Normalization を用いたCNNの定義
class NetInstanceNorm(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1  # 最初の畳み込み層の出力チャネル数

        # 入力はRGB画像（3チャネル）→ 指定チャネル数に変換（例: 32）
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        # Conv1の出力に対してInstance Normalizationを適用
        self.conv1_instancenorm = nn.InstanceNorm2d(num_features=n_chans1)  # 画像/チャネルごとに正規化

        # 2層目の畳み込み：チャネル数を半分に
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        # Conv2の出力に対してInstance Normalizationを適用
        self.conv2_instancenorm = nn.InstanceNorm2d(num_features=n_chans1 // 2)

        # 畳み込み層の出力をフラット化して全結合層へ（出力サイズは8×8×チャネル数）
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        # 最終出力は2クラス（例：飛行機 or 鳥）
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # conv1 → InstanceNorm → Tanh → MaxPooling（2×2）
        out = self.conv1_instancenorm(self.conv1(x))
        out = F.max_pool2d(torch.tanh(out), 2)

        # conv2 → InstanceNorm → Tanh → MaxPooling（2×2）
        out = self.conv2_instancenorm(self.conv2(out))
        out = F.max_pool2d(torch.tanh(out), 2)

        # 特徴マップを1次元ベクトルにフラット化
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)

        # 全結合層 → Tanh活性化
        out = torch.tanh(self.fc1(out))

        # 出力層（Softmaxは後で使用）
        out = self.fc2(out)

        return out

In [ ]:
# バッチ正規化
model = NetBatchNorm(n_chans1=32).to(device=device)  # ここではBN版を選択して学習
# レイヤー正規化
# model = NetLayerNorm(n_chans1=32).to(device=device)
# インスタンス正規化
# model = NetInstanceNorm(n_chans1=32).to(device=device)

optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
model.train()
training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

model.eval()
validate(model, train_loader, val_loader)  # 正規化方式を切り替えて性能比較できる

## 複雑な構造体を学習するためにより深く：深さ

### スキップ接続

ある層のブロックが出力するときに、入力値を追加する。

In [ ]:
# ResNetのようにスキップ接続を利用する
class NetRes(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)  # 3→n_chans1
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3,
                               padding=1)  # n_chans1→n_chans1//2
        self.conv3 = nn.Conv2d(n_chans1 // 2, n_chans1 // 2,
                               kernel_size=3, padding=1)  # 同チャネル数維持で残差構成
        self.fc1 = nn.Linear(4 * 4 * n_chans1 // 2, 32)  # 3回のPoolで空間 32→16→8→4
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.relu(self.conv1(x)), 2)  # Conv1→ReLU→Pool(32→16)
        out = F.max_pool2d(torch.relu(self.conv2(out)), 2) # Conv2→ReLU→Pool(16→8)
        # 入力値を保存する
        out1 = out
        # 活性関数に入力するときに入力値を加算する
        out = F.max_pool2d(torch.relu(self.conv3(out)) + out1, 2) # Conv3→ReLU＋残差→Pool(8→4)
        out = out.view(-1, 4 * 4 * self.n_chans1 // 2)  # Flatten
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
model = NetRes(n_chans1=32).to(device=device)  # 残差接続ありのCNN NetRes を作成し、GPU/CPU(device)へ転送
optimizer = optim.SGD(model.parameters(), lr=1e-2) # 最適化はSGD
loss_fn = nn.CrossEntropyLoss()  # 損失関数にはクロスエントロピー損失を使用

training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

validate(model, train_loader, val_loader)  # 学習後の精度を train/val で評価

## ResBlockを作り、非常にディープなモデルを構築する

畳み込みニューラルネットワークで100層を超える手法については先ほど述べました。ではまともにPyTorchでそのようなネットワークを構築するにはどのようにすればいいのでしょうか。標準的な戦略としては、（Conv2d、BatchNorm2d、ReLU）+ スキップ接続をResBlockとして定義し、forループ内で動的にネットワークを構築する方法があります。

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, n_chans):
        super(ResBlock, self).__init__()
        #  BatchNorm層はバイアスの影響を消し去るため、慣例的にバイアスは省く
        self.conv = nn.Conv2d(n_chans, n_chans, kernel_size=3,
                              padding=1, bias=False)
        self.batch_norm = nn.BatchNorm2d(num_features=n_chans)
        # ResNetの論文で計算されている標準偏差を持つ正規の乱数を用いた
        # カスタマイズの初期化処理 kaiming_normal_initializes を使用する。
        # なお、初期状態では、バッチ正規化処理は平均0で分散0.5の出力分布を
        # 生成するように初期化される。
        torch.nn.init.kaiming_normal_(self.conv.weight,
                                      nonlinearity='relu')  # He初期化
        torch.nn.init.constant_(self.batch_norm.weight, 0.5)
        torch.nn.init.zeros_(self.batch_norm.bias)

    def forward(self, x):
        out = self.conv(x)
        out = self.batch_norm(out)
        out = torch.relu(out)
        return out + x  # 残差加算

In [ ]:
class NetResDeep(nn.Module):
    def __init__(self, n_chans1=32, n_blocks=100):
        super().__init__()
        self.n_chans1 = n_chans1

        # 32x32, RGB → n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        # 16x16 まではチャネル増分用の stem 的な層
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)

        # ResBlock を n_blocks 個並べる（チャネル数は固定）
        n_res_chans = n_chans1 // 2
        self.res_blocks = nn.Sequential(
            *[ResBlock(n_res_chans) for _ in range(n_blocks)]
        )

        # 出力側の畳み込み（あってもなくても良いが、1層足しておく）
        self.conv_out = nn.Conv2d(n_res_chans, n_res_chans, kernel_size=3, padding=1)

        # 全結合層（最終的に 4x4 × n_res_chans まで落とす想定）
        self.fc1 = nn.Linear(4 * 4 * n_res_chans, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # 32x32 → conv1 → ReLU → MaxPool(2) → 16x16
        out = F.max_pool2d(torch.relu(self.conv1(x)), 2)

        # 16x16 → conv2 → ReLU → MaxPool(2) → 8x8
        out = F.max_pool2d(torch.relu(self.conv2(out)), 2)  # チャネル = n_chans1 // 2

        # 8x8 の解像度のまま ResBlock を n_blocks 個通す
        out = self.res_blocks(out)

        # 8x8 → conv_out → ReLU → MaxPool(2) → 4x4
        out = F.max_pool2d(torch.relu(self.conv_out(out)), 2)

        # Flatten
        out = out.view(out.size(0), -1)  # (batch, 4*4*チャネル)

        # 全結合
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out


In [ ]:
# 深い残差ネットワーク（100ブロック）のモデルを定義し、指定デバイス（CPU or GPU）に転送
model = NetResDeep(n_chans1=32, n_blocks=100).to(device=device)

# 最適化アルゴリズムにSGD（確率的勾配降下法）を使用し、学習率は0.003に設定
optimizer = optim.SGD(model.parameters(), lr=3e-3)

# 損失関数にはクロスエントロピー損失を使用（分類問題向け）
loss_fn = nn.CrossEntropyLoss()

# トレーニングループを実行
# ・100エポック学習
# ・指定のモデルとオプティマイザ、損失関数を使用
# ・train_loaderからミニバッチを取得して繰り返し学習
training_loop(
    n_epochs = 100,              # 学習エポック数
    optimizer = optimizer,       # 最適化アルゴリズム（SGD）
    model = model,               # 学習対象モデル（NetResDeep）
    loss_fn = loss_fn,           # 損失関数（クロスエントロピー）
    train_loader = train_loader, # 学習データローダー
)

## 🔧 実践問題1：オプティマイザと学習率を工夫して精度を改善する

上のコードでは `optim.SGD` を学習率 `1e-2` で使用して `NetWidth` を学習しました。
しかし、オプティマイザの種類や学習率の設定によって、収束速度や最終精度は大きく変わります。

**問題：** 以下のコードの `______` 部分を自分で考えて埋め、上の結果より高い検証精度を目指してください。

以下の選択肢を参考に、オプティマイザと学習率の組み合わせを決めてください。

| オプティマイザ | 特徴 |
|:---|:---|
| `optim.SGD` | シンプルだが収束が遅い場合がある。`momentum` 引数を追加すると改善されることが多い |
| `optim.Adam` | 勾配の1次・2次モーメントで学習率を適応的に調整。一般にSGDより小さい学習率が適する |
| `optim.RMSprop` | 勾配の二乗平均で学習率を調整。Adam登場以前に広く使われた手法 |

> **注意：** オプティマイザによって適切な学習率の範囲が異なります。SGDでうまくいった `1e-2` がAdamでもうまくいくとは限りません。

In [ ]:
# オプティマイザと学習率を自分で選んで精度改善に挑戦する
model = NetWidth().to(device=device)

# TODO: オプティマイザの種類と学習率を自分で決めて設定してください
optimizer = optim.______(model.parameters(), ______)

loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

validate(model, train_loader, val_loader)

<details><summary>解答例</summary>

```python
# 例1: Adamを使う場合（SGDより小さい学習率が有効なことが多い）
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 例2: SGDにmomentumを追加する場合
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

# 例3: RMSpropを使う場合
optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
```

Adamは適応的学習率のおかげでSGDより速く収束しやすいですが、学習率を大きくしすぎると発散します。
SGDに `momentum=0.9` を加えると勾配の移動平均で更新方向が安定し、素のSGDより高い精度に到達しやすくなります。
唯一の正解はなく、実際に複数パターン試して比較することが重要です。
</details>

## 🔧 実践問題2：チャネル数・ブロック数・学習率を調整してResNetの精度を改善する

上のコードでは `n_chans1=32`、`n_blocks=100`、学習率 `3e-3` で学習しました。
深い残差ネットワークでは、これらのハイパーパラメータのバランスが精度を左右します。

- **`n_chans1`**（初期チャネル数）を増やすとモデルの幅が広がり表現力が上がるが、計算コストも増える
- **`n_blocks`**（ResBlockの数）を増やすと深さが増すが、必ずしも精度向上につながるとは限らない
- **学習率**はモデルの規模に合わせて調整が必要

---

**問題：** 以下のコードの `______` 部分を自分で決めて、上の結果と異なる構成で学習を行い、精度を比較してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
`n_chans1` を増やすと各層のパラメータ数が二乗で増えるため、`n_blocks` を減らして計算量のバランスを取ると良いです。例えば幅を2倍にしたらブロック数を半分以下にするイメージです。学習率はモデルが大きいほど小さめに設定すると安定します。
</blockquote>

</details>

<br/>

In [ ]:
# ハイパーパラメータを自分で設定して精度改善に挑戦する

# TODO: チャネル数とブロック数を自分で決めてください
model = NetResDeep(n_chans1=______, n_blocks=______).to(device=device)

# TODO: 学習率を自分で決めてください
optimizer = optim.SGD(model.parameters(), lr=______)

loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

validate(model, train_loader, val_loader)

<details><summary>解答例と考え方</summary>

```python
# 例1: 幅を広げてブロック数を減らす（計算量は同程度、表現力の方向が異なる）
model = NetResDeep(n_chans1=64, n_blocks=20).to(device=device)
optimizer = optim.SGD(model.parameters(), lr=1e-3)

# 例2: ブロック数を適度にして学習率を上げる
model = NetResDeep(n_chans1=32, n_blocks=50).to(device=device)
optimizer = optim.SGD(model.parameters(), lr=1e-2)
```

一般に、チャネル数を増やす（幅を広げる）方がブロック数を増やす（深さを増す）よりも効果的な場合があります。
特に今回のようなCIFAR-10の2クラス分類は比較的単純なタスクなので、100ブロックは過剰かもしれません。
また、モデルを大きくした場合は学習率を下げる必要が出てくることが多いです。
逆にモデルを小さくすると、やや大きい学習率でも安定して学習できる傾向があります。
</details>

# 📝 確認演習問題
以下の各問題について、（あ）（い）に当てはまる語句の正しい組み合わせを1〜4から選んでください。

---

## 問題1：畳み込み層の入出力

RGB画像（3チャネル）を入力とし、16種類の特徴を検出したい場合、
PyTorchでは `nn.Conv2d` の第1引数に **（あ）** 、第2引数に **（い）** を指定する。

```
1. （あ）16　（い）3
2. （あ）3　 （い）16
3. （あ）3　 （い）3
4. （あ）16　（い）16
```

<details><summary>正解</summary>

**2.（あ）3（い）16**

`nn.Conv2d` の第1引数は入力チャネル数、第2引数は出力チャネル数です。RGB画像は3チャネルなので第1引数に3、検出したい特徴の数が16なので第2引数に16を指定します。
</details>

---

## 問題2：パディングの役割

カーネルサイズ3×3の畳み込みをパディングなしで適用すると、出力の空間サイズは入力に対して各辺 **（あ）** ピクセルずつ縮小する。
これを防いで入力と同じサイズを維持するには、`padding=（い）` を指定すればよい。

```
1. （あ）1　（い）1
2. （あ）2　（い）2
3. （あ）1　（い）2
4. （あ）2　（い）1
```

<details><summary>正解</summary>

**1.（あ）1（い）1**

カーネルサイズ $k$ の畳み込みでは出力サイズが各辺 $(k-1)/2$ ずつ縮小します。$k=3$ の場合は各辺1ピクセルの縮小なので、`padding=1` で入力と同じサイズを維持できます。
</details>

---

## 問題3：MaxPoolingの効果

CNNでは畳み込み層の後にMaxPoolingを挟むことが一般的である。
`nn.MaxPool2d(2)` を適用すると、空間サイズが **（あ）** になり、
これを繰り返すことで深い層ほど **（い）** な特徴を捉えられるようになる。

```
1. （あ）2倍　　（い）局所的
2. （あ）半分　　（い）局所的
3. （あ）半分　　（い）大域的
4. （あ）2倍　　（い）大域的
```

<details><summary>正解</summary>

**3.（あ）半分（い）大域的**

MaxPool2d(2)は2×2の領域から最大値を取り、空間サイズを半分に縮小します。これを繰り返すことで、各ピクセルが元画像のより広い範囲（受容野）をカバーし、大域的な特徴を捉えられるようになります。
</details>

---

## 問題4：ドロップアウトとバッチ正規化の動作モード

ドロップアウトやバッチ正規化は、訓練時と推論時で異なる挙動をする。
PyTorchでは **（あ）** を呼ぶと訓練モードに、**（い）** を呼ぶと推論モードに切り替わり、
ドロップアウトの無効化やバッチ統計の固定が行われる。

```
1. （あ）model.train()　（い）model.eval()
2. （あ）model.eval()　 （い）model.train()
3. （あ）model.fit()　　（い）model.predict()
4. （あ）model.train()　（い）model.test()
```

<details><summary>正解</summary>

**1.（あ）model.train()（い）model.eval()**

`model.train()` でDropoutが有効・BatchNormが学習用統計を使うモードになり、`model.eval()` でDropoutが無効・BatchNormが固定統計を使う推論モードになります。
</details>

---

## 問題5：残差接続（スキップ接続）の仕組み

ResNetで導入された残差接続では、あるブロックが学習する関数を $F(x)$ としたとき、
ブロックの最終出力は **（あ）** となる。これにより、$F(x)$ は入力と出力の **（い ）** を学習すればよくなり、
非常に深いネットワークでも効率的に学習が進む。

```
1. （あ）$F(x)$　　　　（い）差分（残差）
2. （あ）$F(x) + x$　　（い）差分（残差）
3. （あ）$F(x) + x$　　（い）積
4. （あ）$F(x) \times x$（い）差分（残差）
```

<details><summary>正解</summary>

**2.（あ）$F(x) + x$（い）差分（残差）**

残差接続では出力を $F(x) + x$ とすることで、ブロックは入力 $x$ からの「差分（残差）」のみを学習します。恒等写像がショートカットで保証されるため、勾配消失が起きにくく100層以上のネットワークも学習可能になります。
</details>